# Metrics Visual Summary

Creates a single presentation-style image for performance and cost metrics.


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from matplotlib.ticker import FuncFormatter

NOTEBOOK_DIR_CANDIDATES = [Path.cwd(), Path.cwd() / 'macro_regime_v6_final' / 'macro_regime_final' / 'notebooks']
NOTEBOOK_DIR = next((p for p in NOTEBOOK_DIR_CANDIDATES if (p / 'metrics_visual_summary.ipynb').exists()), None)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError('Could not resolve notebook directory for metrics_visual_summary.ipynb')

BASE = NOTEBOOK_DIR.parent
REPORTS = BASE / 'reports'
UNLEV_DIR = BASE / 'report_snapshots' / 'unlevered_profile_2026-04-06'
OUT_DIR = BASE / 'reports'

bt = pd.read_csv(REPORTS / 'backtest_results.csv')
bt['Date'] = pd.to_datetime(bt['Date'])
bt = bt.sort_values('Date').reset_index(drop=True)
bt['StrategyRet'] = bt['PortRetKS'] if 'PortRetKS' in bt.columns else bt['PortRet']
bt['StrategyEq'] = (1 + bt['StrategyRet']).cumprod()
bt['SPYEq'] = (1 + bt['SPYRet']).cumprod()

unlev = pd.read_csv(UNLEV_DIR / 'backtest_results.csv')
unlev['Date'] = pd.to_datetime(unlev['Date'])
unlev = unlev.sort_values('Date').reset_index(drop=True)
unlev['StrategyRet'] = unlev['PortRetKS'] if 'PortRetKS' in unlev.columns else unlev['PortRet']
unlev['StrategyEq'] = (1 + unlev['StrategyRet']).cumprod()

def cagr(r, periods=4):
    r = pd.Series(r).dropna()
    if r.empty:
        return np.nan
    total = float((1 + r).prod())
    years = len(r) / periods
    return total ** (1 / years) - 1 if years else np.nan

def ann_vol(r, periods=4):
    r = pd.Series(r).dropna()
    if len(r) < 2:
        return np.nan
    return float(r.std(ddof=1) * np.sqrt(periods))

def max_dd(eq):
    eq = pd.Series(eq).dropna()
    return np.nan if eq.empty else float((eq / eq.cummax() - 1).min())

def sharpe(r):
    cg = cagr(r)
    v = ann_vol(r)
    return np.nan if pd.isna(v) or v == 0 else cg / v

def sortino(r, periods=4):
    r = pd.Series(r).dropna()
    downside = r[r < 0]
    if len(downside) < 2:
        return np.nan
    dv = float(downside.std(ddof=1) * np.sqrt(periods))
    cg = cagr(r, periods)
    return np.nan if dv == 0 else cg / dv

def hit_rate(r):
    r = pd.Series(r).dropna()
    return float((r > 0).mean()) if not r.empty else np.nan

def fmt_pct(x):
    return '' if pd.isna(x) else f'{x:.2%}'

def fmt_num(x):
    return '' if pd.isna(x) else f'{x:.2f}'

rows = [
    {
        'Profile': 'Levered',
        'Total Return': (1 + bt['StrategyRet']).prod() - 1,
        'CAGR': cagr(bt['StrategyRet']),
        'Ann Vol': ann_vol(bt['StrategyRet']),
        'Sharpe': sharpe(bt['StrategyRet']),
        'Sortino': sortino(bt['StrategyRet']),
        'Max DD': max_dd(bt['StrategyEq']),
        'Hit Rate': hit_rate(bt['StrategyRet']),
        'Exec Cost': bt['TxnCostBps'].mean(),
        'Financing Cost': bt['FinancingCostRet'].mean() * 1e4,
    },
    {
        'Profile': 'Unlevered',
        'Total Return': (1 + unlev['StrategyRet']).prod() - 1,
        'CAGR': cagr(unlev['StrategyRet']),
        'Ann Vol': ann_vol(unlev['StrategyRet']),
        'Sharpe': sharpe(unlev['StrategyRet']),
        'Sortino': sortino(unlev['StrategyRet']),
        'Max DD': max_dd(unlev['StrategyEq']),
        'Hit Rate': hit_rate(unlev['StrategyRet']),
        'Exec Cost': unlev['TxnCostBps'].mean(),
        'Financing Cost': unlev['FinancingCostRet'].mean() * 1e4,
    },
    {
        'Profile': 'SPY',
        'Total Return': (1 + bt['SPYRet']).prod() - 1,
        'CAGR': cagr(bt['SPYRet']),
        'Ann Vol': ann_vol(bt['SPYRet']),
        'Sharpe': sharpe(bt['SPYRet']),
        'Sortino': sortino(bt['SPYRet']),
        'Max DD': max_dd(bt['SPYEq']),
        'Hit Rate': hit_rate(bt['SPYRet']),
        'Exec Cost': np.nan,
        'Financing Cost': np.nan,
    },
]

metrics = pd.DataFrame(rows)
metrics


In [ ]:
fig = plt.figure(figsize=(18, 10), facecolor='#f8fafc')
gs = fig.add_gridspec(2, 2, height_ratios=[1.1, 1], width_ratios=[1.15, 1], hspace=0.18, wspace=0.12)

ax_title = fig.add_subplot(gs[0, 0])
ax_bar = fig.add_subplot(gs[0, 1])
ax_cost = fig.add_subplot(gs[1, 0])
ax_eq = fig.add_subplot(gs[1, 1])

for ax in [ax_title, ax_bar, ax_cost, ax_eq]:
    ax.set_facecolor('white')

ax_title.axis('off')
ax_title.text(0.0, 0.97, 'Backtest Performance And Costs', fontsize=26, fontweight='bold', color='#0f172a', va='top')
ax_title.text(0.0, 0.89, 'Net of execution costs; levered profile also includes financing cost.', fontsize=13, color='#475569', va='top')

cards = [
    ('Levered CAGR', fmt_pct(metrics.loc[metrics['Profile'] == 'Levered', 'CAGR'].iloc[0])),
    ('Unlevered CAGR', fmt_pct(metrics.loc[metrics['Profile'] == 'Unlevered', 'CAGR'].iloc[0])),
    ('Levered Sharpe', fmt_num(metrics.loc[metrics['Profile'] == 'Levered', 'Sharpe'].iloc[0])),
    ('Financing Cost', f"{metrics.loc[metrics['Profile'] == 'Levered', 'Financing Cost'].iloc[0]:.2f} bps/q"),
]

positions = [(0.00, 0.52), (0.50, 0.52), (0.00, 0.18), (0.50, 0.18)]
for (title, value), (x, y) in zip(cards, positions):
    rect = FancyBboxPatch((x, y), 0.42, 0.22, boxstyle='round,pad=0.012,rounding_size=0.03', facecolor='#eef2ff', edgecolor='#c7d2fe')
    ax_title.add_patch(rect)
    ax_title.text(x + 0.03, y + 0.16, title, fontsize=11, color='#64748b', fontweight='bold')
    ax_title.text(x + 0.03, y + 0.06, value, fontsize=22, color='#111827', fontweight='bold')

show = metrics.copy()
for c in ['Total Return', 'CAGR', 'Ann Vol', 'Max DD', 'Hit Rate']:
    show[c] = show[c].map(fmt_pct)
for c in ['Sharpe', 'Sortino']:
    show[c] = show[c].map(fmt_num)
show['Exec Cost'] = show['Exec Cost'].map(lambda x: '' if pd.isna(x) else f'{x:.2f} bps')
show['Financing Cost'] = show['Financing Cost'].map(lambda x: '' if pd.isna(x) else f'{x:.2f} bps')

table = ax_title.table(
    cellText=show[['Profile', 'Total Return', 'CAGR', 'Sharpe', 'Sortino', 'Max DD']].values,
    colLabels=['Profile', 'Total Return', 'CAGR', 'Sharpe', 'Sortino', 'Max DD'],
    cellLoc='center',
    bbox=[0.0, -0.10, 0.98, 0.24]
)
table.auto_set_font_size(False)
table.set_fontsize(10)

bar_df = metrics[metrics['Profile'].isin(['Levered', 'Unlevered', 'SPY'])]
x = np.arange(len(bar_df))
w = 0.35
ax_bar.bar(x - w/2, bar_df['CAGR'], width=w, color=['#0f766e', '#b45309', '#334155'], label='CAGR')
ax_bar.bar(x + w/2, bar_df['Ann Vol'], width=w, color=['#5eead4', '#fdba74', '#94a3b8'], label='Ann Vol')
ax_bar.set_xticks(x)
ax_bar.set_xticklabels(bar_df['Profile'])
ax_bar.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'{y:.0%}'))
ax_bar.set_title('Return vs Risk')
ax_bar.legend(loc='upper right')

cost_df = metrics[metrics['Profile'].isin(['Levered', 'Unlevered'])][['Profile', 'Exec Cost', 'Financing Cost']].set_index('Profile')
cost_df.plot(kind='bar', ax=ax_cost, color=['#b45309', '#7c3aed'])
ax_cost.set_title('Average Quarterly Costs')
ax_cost.set_ylabel('bps')
ax_cost.legend(loc='upper right')
ax_cost.tick_params(axis='x', rotation=0)

ax_eq.plot(bt['Date'], bt['StrategyEq'], color='#0f766e', linewidth=2.5, label='Levered')
ax_eq.plot(unlev['Date'], unlev['StrategyEq'], color='#b45309', linewidth=2.5, label='Unlevered')
ax_eq.plot(bt['Date'], bt['SPYEq'], color='#334155', linewidth=2.0, label='SPY')
ax_eq.set_title('Net Equity Curves')
ax_eq.set_ylabel('Growth of $1')
ax_eq.legend(loc='upper left')

fig.suptitle('Macro Regime Strategy Visual Summary', fontsize=30, fontweight='bold', color='#0f172a', y=0.98)
fig.subplots_adjust(left=0.05, right=0.98, top=0.91, bottom=0.06)

out_path = OUT_DIR / 'metrics_visual_summary.png'
fig.savefig(out_path, dpi=160, bbox_inches='tight', facecolor=fig.get_facecolor())
print(f'Saved image to: {out_path}')
plt.close(fig)
